# Publication figures — Coherence-Rate Graphs for Noisy Equivariant Quantum Neural Networks

This notebook renders the seven publication figures of the paper from the
CSV files written by the pipeline notebook. It reads the results directory
only and performs no computation of its own, so the figures can be
regenerated, restyled or extended without touching any numerical output.

Set `OUT_DIR` in the configuration cell below to the directory holding the
phase CSVs (the repository's `results/` directory as committed, or the
output directory of your own pipeline run); figures are written to its
`figures/` subdirectory.


In [ ]:
# ============================== Configuration ==============================
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# Location of the phase CSVs. On Google Colab, mount Drive and point this
# at your results folder; locally, the repository's results/ directory
# works as committed.
try:
    from google.colab import drive  # noqa: F401
    drive.mount("/content/drive", force_remount=False)
    OUT_DIR = Path("/content/drive/MyDrive") / "QNN-Coherence-Rate-Graph/results"
except ImportError:
    OUT_DIR = Path.cwd() / "results"

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"reading CSVs from {OUT_DIR}\nwriting figures to {FIG_DIR}")


## Figure style

All figures are rendered from the cached CSVs only. The style cell below
fixes fonts, sizes and resolution for the journal layout.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300,
    "font.size": 9, "axes.titlesize": 10, "axes.titlepad": 8,
    "axes.labelsize": 9.5, "legend.fontsize": 8,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "mathtext.fontset": "cm", "axes.axisbelow": True,
})
CH_COLORS = {"amp_damp": "#1f77b4", "dephase": "#ff7f0e",
             "depol": "#2ca02c", "x_err": "#d62728",
             "inhom_dephase": "#9467bd", "site_amp_damp": "#8c564b",
             "biased_pauli": "#bcbd22", "corr_dephase": "#111111",
             "coh_diss_mix": "#7f7f7f", "site_z_overrotation": "#e377c2"}
CH_LABELS = {"amp_damp": "Amplitude damping", "dephase": "Dephasing",
             "depol": "Depolarising", "x_err": "X error",
             "inhom_dephase": "Inhomogeneous dephasing",
             "site_amp_damp": "Site-dependent damping",
             "biased_pauli": "Biased Pauli",
             "corr_dephase": "Correlated dephasing",
             "coh_diss_mix": "Coherent-dissipative mix",
             "site_z_overrotation": "Site Z over-rotation"}
CH_ORDER = ["amp_damp", "dephase", "depol", "x_err", "biased_pauli",
            "inhom_dephase", "site_amp_damp", "corr_dephase",
            "coh_diss_mix"]
def nice_grid(ax):
    ax.grid(True, which="major", linewidth=0.4, alpha=0.28)
def savefig(fig, name):
    p = FIG_DIR / name
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"figure -> {p}")

### Figure 1 — coherence-rate graph recovery

In [ ]:
pr = pd.read_csv(OUT_DIR / "phaseB_pair_rates.csv")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7), constrained_layout=True)
handles = []
for name in CH_ORDER:
    g = pr[pr.channel == name]
    if not len(g):
        continue
    h, = ax[0].plot(g.rate_analytic, g.rate_numeric, "o", ms=3.6,
                    alpha=0.65, mew=0, color=CH_COLORS[name],
                    label=CH_LABELS[name])
    handles.append(h)
lim = [-0.2, 1.06 * pr.rate_analytic.max()]
ax[0].plot(lim, lim, "--", lw=0.9, color="0.55", zorder=0)
ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel("Analytic first-order rate")
ax[0].set_ylabel("Numeric restricted-generator rate")
ax[0].set_title("(a) Coherence-graph rate recovery")
nice_grid(ax[0])
for name in CH_ORDER:
    g = pr[pr.channel == name]
    if not len(g):
        continue
    ax[1].plot(np.sort(g.rate_numeric.values), color=CH_COLORS[name],
               lw=1.4)
ax[1].set_xlabel("Pair index (sorted)")
ax[1].set_ylabel("Pairwise rate")
ax[1].set_title("(b) Rate spectra: flat, structured and protected")
nice_grid(ax[1])
fig.legend(handles=handles, loc="outside lower center", ncol=5,
           frameon=False, columnspacing=1.1, handletextpad=0.4)
savefig(fig, "fig1_graph_recovery.png")

### Figure 2 — cut-crossing predictor vs measurement

In [ ]:
c = pd.read_csv(OUT_DIR / "phaseC_cut_crossing.csv")
cg = (c.groupby(["depth", "ell", "q"])
        .agg(pred=("omega_resp_pred", "mean"),
             meas=("omega_hat_measured", "mean"),
             prot=("cut_predictor_protected", "first"),
             vis=("visible_unprotected_fraction", "mean")).reset_index())
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7), constrained_layout=True)
x = np.arange(len(cg))
top = float(max(cg.pred.max(), cg.meas.max(), 0.05))
ax[0].bar(x - 0.19, cg.pred, width=0.38, color="#1f77b4",
          label=r"Predicted $\omega^{\mathrm{resp}}$")
ax[0].bar(x + 0.19, cg.meas, width=0.38, color="#ff7f0e",
          label=r"Measured $\hat{\omega}$ (finite $\gamma$, mean)")
# linear gamma -> 0 extrapolation where two strengths are available
ex_x, ex_y = [], []
for xi, ((d, e_, q_), grp) in enumerate(c.groupby(["depth", "ell", "q"])):
    grp = grp.sort_values("gamma")
    if grp.gamma.nunique() >= 2 and not grp.cut_predictor_protected.iloc[0]:
        om = grp.groupby("gamma").omega_hat_measured.mean()
        ex_x.append(xi)
        ex_y.append(2 * om.iloc[0] - om.iloc[1])
if ex_x:
    ax[0].plot(np.array(ex_x) + 0.19, ex_y, "o", ms=5, mfc="white",
               mec="#ff7f0e", mew=1.3, zorder=5,
               label=r"$\gamma \to 0$ extrapolation")
for xi, (p, pv, mv) in enumerate(zip(cg.prot, cg.pred, cg.meas)):
    y = max(pv, mv, 0.0) + 0.07 * top
    ax[0].text(xi, y, "P" if p else "E", ha="center", va="bottom",
               fontsize=9, fontweight="bold",
               color="#1a7d1a" if p else "#c22040")
ax[0].set_ylim(-0.02 * top, 1.42 * top)
ax[0].set_xticks(x)
ax[0].set_xticklabels([f"$L={d}$\n({e},{q})" for d, e, q
                       in zip(cg.depth, cg.ell, cg.q)], fontsize=7.6)
ax[0].set_xlabel("Depth and parameter location")
ax[0].set_ylabel("Alignment ratio")
ax[0].set_title("(a) Cut predictor (P/E) against response")
ax[0].legend(loc="upper left", frameon=False)
nice_grid(ax[0])
ax[1].plot(cg.vis, cg.meas, "o", ms=5.5, color="#333333", mew=0)
ax[1].set_xlim(-0.06, 1.06)
ax[1].set_ylim(-0.06 * max(top, 0.4), 1.18 * max(top, 0.4))
ax[1].set_xlabel("Visible weight on unprotected pairs")
ax[1].set_ylabel(r"Measured $\hat{\omega}$")
ax[1].set_title("(b) Visible cut mass tracks measured exposure")
nice_grid(ax[1])
savefig(fig, "fig2_cut_crossing.png")

### Figure 3 — support-cut bound and adversarial sweep

In [ ]:
d1 = pd.read_csv(OUT_DIR / "phaseD_bound.csv")
d2 = pd.read_csv(OUT_DIR / "phaseD_sweep.csv")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7), constrained_layout=True)
for name in CH_ORDER:
    g = d1[d1.channel == name]
    if not len(g):
        continue
    ax[0].plot(g.support_cut_bound, g.lambda_mode, "o", ms=5, mew=0,
               alpha=0.85, color=CH_COLORS[name], label=CH_LABELS[name])
m = 1.08 * float(d1.lambda_mode.max())
ax[0].plot([0, m], [0, m], "--", lw=0.9, color="0.55", zorder=0)
ax[0].set_xlim(-0.02 * m, m); ax[0].set_ylim(-0.02 * m, m)
ax[0].set_xlabel(r"Support lower bound $\phi\, r_{\min}$")
ax[0].set_ylabel(r"Mode rate $\lambda_{\mathrm{mode}}$")
ax[0].set_title("(a) Support bound audit")
ax[0].legend(loc="lower right", frameon=False, fontsize=7.2)
nice_grid(ax[0])
KIND_LABELS = {"dephasing_profile": "Random dephasing profiles",
               "zz_edge_set": "Random ZZ edge sets"}
for kind, g in d2.groupby("kind"):
    ax[1].plot(g.omega_resp, g.omega_mode, "o", ms=4.5, mew=0, alpha=0.75,
               label=KIND_LABELS.get(kind, kind))
ax[1].plot([0, 1], [0, 1], "--", lw=0.9, color="0.55", zorder=0)
ax[1].set_xlim(-0.04, 1.04); ax[1].set_ylim(-0.04, 1.04)
ax[1].set_xlabel(r"Response alignment $\omega^{\mathrm{resp}}$")
ax[1].set_ylabel(r"Mode alignment $\omega^{\mathrm{mode}}$")
ax[1].set_title("(b) Adversarial sweep: mode against response")
ax[1].legend(loc="lower right", frameon=False)
nice_grid(ax[1])
savefig(fig, "fig3_bound.png")

### Figure 4 — slot closed forms and the two-term predictor

In [ ]:
e = pd.read_csv(OUT_DIR / "phaseE_slot_forms.csv")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7), constrained_layout=True)
import ast
SLOT_COLOURS = ["#1f77b4", "#ff7f0e", "#2ca02c"]
n_series = len(e)
width, gap = 0.24, 0.03
max_slot = 0
for k, (_, r) in enumerate(e.iterrows()):
    per = ast.literal_eval(r.per_slot_over_lambda)
    ks = sorted(per); vs = [per[kk] for kk in ks]
    max_slot = max(max_slot, max(ks))
    off = (k - (n_series - 1) / 2) * (width + gap)
    lab = (f"{CH_LABELS.get(r.channel, r.channel)}, $L={r.depth}$ "
           rf"($\omega^{{\mathrm{{resp}}}}={r.omega_resp:.3f}$)")
    ax[0].bar(np.array(ks) + off, vs, width=width,
              color=SLOT_COLOURS[k % 3], label=lab)
ax[0].axhline(1.0, color="0.55", lw=0.9, ls="--")
ax[0].axhline(0.0, color="0.35", lw=0.9)
ax[0].set_ylim(-0.08, 1.78)
ax[0].set_yticks([0.0, 0.5, 1.0])
ax[0].set_xticks(np.arange(max_slot + 1))
ax[0].set_xlim(-0.65, max_slot + 0.65)
ax[0].set_xlabel("Noise slot $\\ell$")
ax[0].set_ylabel(r"$r_\ell\,/\,\lambda_{\mathrm{coh}}$")
ax[0].set_title("(a) Slot structure: dephasing final slot pairs to zero")
ax[0].legend(loc="upper center", frameon=False, fontsize=7.6,
             borderaxespad=0.4)
nice_grid(ax[0])
f2g = pd.read_csv(OUT_DIR / "phaseF2_uniform_grid.csv")
for name in ("dephase", "amp_damp", "corr_dephase"):
    g = f2g[f2g.channel == name]
    if not len(g):
        continue
    col = CH_COLORS[name]
    g = g.sort_values("gamma")
    ax[1].plot(g.gamma, g.ratio_measured, "o", ms=4.5, mew=0, color=col)
    ax[1].plot(g.gamma, g.ratio_exact, "-", lw=1.6, color=col,
               label=CH_LABELS[name])
ax[1].set_ylim(-0.02, 1.06)
ax[1].set_xlabel(r"Per-layer strength $\gamma$")
ax[1].set_ylabel(r"$M_2(\gamma)\,/\,M_2(0)$")
ax[1].set_title("(b) Exact per-slot product laws against measurement")
ax[1].legend(loc="lower left", frameon=False)
nice_grid(ax[1])
savefig(fig, "fig4_slots_second_order.png")

### Figure 5 — higher-sector protection collapse and residual scaling

In [ ]:
g2 = pd.read_csv(OUT_DIR / "phaseG_r2_graph.csv")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7), constrained_layout=True)
gg = g2[~g2.channel.str.endswith("_reference")]
ax[0].bar(np.arange(len(gg)), 100 * gg.protected_fraction,
          color=[CH_COLORS.get(cn, "k") for cn in gg.channel], width=0.62)
ref = g2[g2.channel.str.endswith("_reference")]
ymax = 100 * float(gg.protected_fraction.max())
if len(ref):
    rv = 100 * float(ref.protected_fraction.iloc[0])
    ax[0].axhline(rv, color="k", lw=1.0, ls="--",
                  label=f"Correlated control, $r=1$ ({rv:.1f}%)")
    ymax = max(ymax, rv)
ax[0].set_ylim(0, 1.45 * max(ymax, 1.0))
ax[0].set_xticks(np.arange(len(gg)))
ax[0].set_xticklabels([CH_LABELS.get(cn, cn) for cn in gg.channel],
                      rotation=30, ha="right", fontsize=7.4)
ax[0].set_ylabel("Protected pairs (%)")
ax[0].set_title("(a) Protected-pair counts at $r=2$ (graph statistic; Phase K2)")
ax[0].legend(loc="upper right", frameon=False)
nice_grid(ax[0])
fd = pd.read_csv(OUT_DIR / "phaseF2_fd_convergence.csv")
for name, g in fd.groupby("channel"):
    g = g.sort_values("h")
    col = CH_COLORS.get(name, "k")
    ax[1].loglog(g.h, g.rel_error, "-o", ms=4, lw=1.4, color=col,
                 label=CH_LABELS.get(name, name))
hh = np.array(sorted(fd.h.unique()))
if len(hh) >= 2:
    ref = fd.rel_error.max() * hh / hh.max()
    ax[1].loglog(hh, ref, "--", lw=0.9, color="0.55", label="Slope 1")
ax[1].set_xlabel("Finite-difference step $h$")
ax[1].set_ylabel(r"Relative error of $K_2$ estimate")
ax[1].set_title("(b) Finite differences converge to the exact coefficients")
ax[1].legend(loc="upper left", frameon=False)
nice_grid(ax[1])
savefig(fig, "fig5_r2_and_residuals.png")

### Figure 6 — deep-circuit stress test

In [ ]:
h = pd.read_csv(OUT_DIR / "phaseH_deep_stress.csv")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7), constrained_layout=True)
stress_top = int(h.depth.max())
hs = h[h.depth == stress_top]
PARAM_COLOURS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]
gfine = np.linspace(0, float(h.gamma.max()) * 1.02, 120)
for k, ((e_, q_), g) in enumerate(hs.groupby(["ell", "q"])):
    col = PARAM_COLOURS[k % 4]
    g = g.sort_values("gamma")
    lab = f"$L={stress_top}$, parameter ({e_},{q_})"
    ax[0].errorbar(g.gamma, g.om_hat,
                   yerr=[g.om_hat - g.om_hat_lo, g.om_hat_hi - g.om_hat],
                   fmt="o", ms=4.5, lw=1.1, capsize=2.5, color=col,
                   label=lab)
    coef = np.polyfit(g.gamma, g.om_hat, 2)
    ax[0].plot(gfine, np.polyval(coef, gfine), "-", lw=1.2, color=col,
               alpha=0.75)
    ax[0].errorbar([0.0], [g.om_extrap.iloc[0]],
                   yerr=[[g.om_extrap.iloc[0] - g.extrap_lo.iloc[0]],
                         [g.extrap_hi.iloc[0] - g.om_extrap.iloc[0]]],
                   fmt="D", ms=5, lw=1.2, capsize=2.5, color=col, mew=0)
    ax[0].axhspan(g.pred_lo.iloc[0], g.pred_hi.iloc[0], color=col,
                  alpha=0.12, lw=0)
    ax[0].axhline(g.omega_pred.iloc[0], color=col, lw=1.0, ls="--",
                  alpha=0.8)
ax[0].set_xlim(-0.0012, float(h.gamma.max()) * 1.06)
ax[0].set_xticks([0.0] + sorted(h.gamma.unique()))
ax[0].set_xticklabels(["0"] + [f"{g:g}" for g in sorted(h.gamma.unique())])
ax[0].set_xlabel(r"Per-layer strength $\gamma$")
ax[0].set_ylabel(r"Alignment $\hat{\omega}$")
ax[0].set_title(f"(a) Depth-{stress_top} extrapolation against prediction")
ax[0].legend(loc="lower left", frameon=False)
nice_grid(ax[0])
h2 = pd.read_csv(OUT_DIR / "phaseH2_equivalence.csv")
h2 = h2.sort_values(["depth", "ell", "q"]).reset_index(drop=True)
xs = np.arange(len(h2))
mrg = float(h2.margin.iloc[0])
ax[1].axhspan(-mrg, mrg, color="#2ca02c", alpha=0.12, lw=0,
              label=f"Equivalence margin $\\pm{mrg:g}$")
ax[1].axhline(0.0, color="0.4", lw=0.8)
ax[1].errorbar(xs, h2.diff_median,
               yerr=[h2.diff_median - h2.diff_ci95_lo,
                     h2.diff_ci95_hi - h2.diff_median],
               fmt="none", ecolor="#bbbbbb", elinewidth=3.2, capsize=0)
ax[1].errorbar(xs, h2.diff_median,
               yerr=[h2.diff_median - h2.diff_ci90_lo,
                     h2.diff_ci90_hi - h2.diff_median],
               fmt="D", ms=5, lw=1.4, capsize=2.5, color="#d62728", mew=0,
               label="Difference, median with 90% (95% grey) CI")
ymax = float(max(h2.diff_ci95_hi.max(), mrg) * 1.35)
ymin = float(min(h2.diff_ci95_lo.min(), -mrg) * 1.35)
for x_, v in zip(xs, h2.verdict):
    ax[1].text(x_, ymax * 0.88, v, ha="center", fontsize=6.8, color="0.25")
ax[1].set_ylim(ymin, ymax)
ax[1].set_xticks(xs)
ax[1].set_xticklabels([f"$L={d}$\n({e_},{q_})" for d, e_, q_
                       in zip(h2.depth, h2.ell, h2.q)], fontsize=7.6)
ax[1].set_xlabel("Depth and parameter location")
ax[1].set_ylabel(r"$\hat{\omega}_{\gamma\to0} - \omega^{\mathrm{resp}}$")
ax[1].set_title("(b) Difference-based verdicts (Phase H2)")
ax[1].legend(loc="lower left", frameon=False, fontsize=7)
nice_grid(ax[1])
savefig(fig, "fig6_deep_stress.png")

### Figure 7 — factorial and population benchmark

In [ ]:
k2 = pd.read_csv(OUT_DIR / "phaseK2_factorial.csv")
pp = pd.read_csv(OUT_DIR / "phaseP_benchmark.csv")
ps = pd.read_csv(OUT_DIR / "phaseP_estimator_stats.csv")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.7), constrained_layout=True)
kk = k2[k2.ell >= 0].reset_index(drop=True)
xs = np.arange(len(kk))
cols = ["#1f77b4" if r == 1 else "#d62728" for r in kk.sector_r]
ax[0].bar(xs, kk.support_weight_protected_fraction, width=0.55, color=cols)
for x_, h_, dv in zip(xs, kk.support_weight_protected_fraction,
                      kk.delta_tilde_g002):
    ax[0].text(x_, h_ + 0.035, rf"$\tilde\delta={dv:+.3f}$", ha="center",
               fontsize=6.8, color="0.2")
miss = k2[k2.ell < 0]
labels = [f"$r={r}$\n{i}\n({e_},{q_})" for r, i, e_, q_
          in zip(kk.sector_r, kk.input, kk.ell, kk.q)]
ax[0].set_xticks(xs); ax[0].set_xticklabels(labels, fontsize=6.8)
ax[0].set_ylim(0, 1.22)
ax[0].set_yticks([0.0, 0.5, 1.0])
ax[0].set_ylabel("Support-weight protected fraction")
ax[0].set_title("(a) Factorial control layout "
                + ("(localised $r=2$: structurally inactive)"
                   if len(miss) else ""))
nice_grid(ax[0])
Lmax = int(pp.depth.max())
stress = sorted(set(zip(pp[pp.depth == Lmax].ell, pp[pp.depth == Lmax].q)))
xoff = 0.0
sizes = sorted(ps[ps.record_type == "replicate_mean"]
               .estimator_size.unique())
tick_pos, tick_lab = [], []
first = True
for (e_, q_) in stress:
    prow = pp[(pp.depth == Lmax) & (pp.ell == e_) & (pp.q == q_)]
    assert len(prow) == 1, "population row join must be unique"
    prow = prow.iloc[0]
    sub = ps[(ps.record_type == "replicate_mean") & (ps.depth == Lmax)
             & (ps.ell == e_) & (ps.q == q_)].sort_values("estimator_size")
    if not len(sub):
        continue
    xs2 = np.array([xoff + k for k in range(len(sub))])
    ax[1].errorbar(xs2, sub.estimate,
                   yerr=[sub.estimate - sub.ci95_lo,
                         sub.ci95_hi - sub.estimate],
                   fmt="o", ms=5, capsize=3, lw=1.3, mew=0,
                   label=f"Replicate means ({e_},{q_}), "
                         f"{sub.interval_kind.iloc[0]} CI")
    ax[1].hlines(prow.omega_population, xoff - 0.35,
                 xoff + len(sub) - 0.65, color="0.15", lw=1.4)
    mrow = ps[(ps.record_type == "measured_extrapolation")
              & (ps.depth == Lmax) & (ps.ell == e_) & (ps.q == q_)]
    if len(mrow):
        ax[1].plot([xoff + len(sub) - 0.45], mrow.estimate, "D",
                   ms=6, color="#d62728", mew=0)
    for k, sz in enumerate(sub.estimator_size):
        tick_pos.append(xoff + k); tick_lab.append(f"$n={int(sz)}$")
    tick_pos.append(xoff + len(sub) - 0.45); tick_lab.append("extrap")
    xoff += len(sub) + 1.0
    first = False
ax[1].plot([], [], color="0.15", lw=1.4, label="Population target")
ax[1].plot([], [], "D", ms=6, color="#d62728", mew=0,
           label=r"Extrapolated $\hat\omega$ (analytic norm.)")
ax[1].set_xticks(tick_pos)
ax[1].set_xticklabels(tick_lab, fontsize=7)
ax[1].set_ylabel("Response alignment")
ax[1].set_title("(b) Estimators against the population target")
ax[1].legend(loc="lower left", frameon=False, fontsize=6.6)
nice_grid(ax[1])
savefig(fig, "fig7_population.png")